# Classic ML Test Majority Vote Metrics (one model per ESP)

Notebook повторяет train/test pipeline из `classic_ml_train.ipynb`, но обучает **отдельную модель для каждой ESP-платы** (`dev1`, `dev2`, `dev3`).

Для каждой ESP выполняется свой train/test split по папкам `test_N`, считаются свои preprocessing-объекты и сохраняются отдельные артефакты.

Метрики считаются только на test split, отдельно по каждой плате.

In [1]:
from pathlib import Path
import json
import re
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

project_root = Path.cwd()
if not (project_root.parent / 'tools').exists() and (project_root.parent.parent / 'tools').exists():
    project_root = project_root.parent.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.csi_parser import Parser
from tools.filters import median_filter
from tools.classic_ml_inference import run_inference

dataset_root = project_root / 'wifi_data_set_fixed_one_person'
median_width = 5
random_state = 42

if median_width % 2 == 0:
    raise ValueError('median_width must be odd')

np.random.seed(random_state)

print(f'project_root: {project_root}')
print(f'dataset_root: {dataset_root}')

project_root: /home/gleb/learning/CSI-activity-detection
dataset_root: /home/gleb/learning/CSI-activity-detection/wifi_data_set_fixed_one_person


## Helpers

Эти функции повторяют preprocessing из `classic_ml_train.ipynb` и добавляют извлечение `esp_id` из имени файла (`...dev1...`, `...dev2...`, `...dev3...`).

In [2]:
def extract_esp_id(file_path: Path) -> str:
    stem = file_path.stem
    match = re.search(r'dev(\d+)', stem, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f'Could not extract ESP id from file name: {file_path.name}')
    return f'dev{int(match.group(1))}'


def parse_path_meta(file_path: Path) -> dict:
    parts = file_path.parts
    return {
        'person_id': next((p for p in parts if p.startswith('id_person_')), 'unknown'),
        'label': next((p for p in parts if p.startswith('label_')), 'unknown'),
        'test_id': next((p for p in parts if p.startswith('test_')), 'unknown'),
        'esp_id': extract_esp_id(file_path) if file_path.suffix == '.data' else 'unknown',
    }


def get_test_group_dir(file_path: Path) -> Path:
    for parent in file_path.parents:
        if parent.name.startswith('test_'):
            return parent
    raise ValueError(f'Could not find test_* folder in path: {file_path}')


def skewness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 3))


def kurtosis_excess(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 4) - 3.0)


def spectral_features(signal: np.ndarray) -> tuple[float, float, float, float, float]:
    x = np.asarray(signal, dtype=np.float64)
    n = len(x)
    if n < 2:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    spectrum = np.fft.rfft(x)
    power = np.abs(spectrum) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0)

    power_sum = power.sum()
    if power_sum <= 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    centroid = float((freqs * power).sum() / power_sum)
    spread = float(np.sqrt(((freqs - centroid) ** 2 * power).sum() / power_sum))
    p_norm = power / power_sum
    entropy = float(-(p_norm * np.log2(p_norm + 1e-12)).sum())
    dominant_freq = float(freqs[int(np.argmax(power))])
    rolloff = float(freqs[np.searchsorted(np.cumsum(power), 0.85 * power_sum)])

    return centroid, spread, entropy, dominant_freq, rolloff


def build_unit_signal(df: pd.DataFrame, window: int) -> np.ndarray:
    amp_matrix = np.stack(df['amplitude'].to_numpy(), axis=0).astype(np.float64)
    raw_time_signal = amp_matrix.mean(axis=1)
    return median_filter(raw_time_signal, window)


def build_unit_cache(file_list: list[Path]) -> list[dict]:
    cache = []
    for file_path in file_list:
        df = Parser(file_path).parse()
        filtered_signal = build_unit_signal(df, median_width)
        cache.append({
            'file_path': file_path,
            'df': df,
            'filtered_signal': filtered_signal,
        })
    return cache

## Group train/test split per ESP

Split делается по папкам `test_N`, но **отдельно для каждой ESP**: у каждой платы свой train/test набор групп.

In [3]:
all_files = sorted(dataset_root.rglob('*.data'))
if not all_files:
    raise FileNotFoundError(f'No .data files found under {dataset_root}')

file_rows = []
for fp in all_files:
    meta = parse_path_meta(fp)
    file_rows.append({
        'file_path': fp,
        'group_path': get_test_group_dir(fp),
        'label': meta['label'],
        'esp_id': meta['esp_id'],
    })

files_df = pd.DataFrame(file_rows)
esp_ids = sorted(files_df['esp_id'].unique())

esp_splits: dict[str, dict] = {}
for esp_id in esp_ids:
    esp_df = files_df[files_df['esp_id'] == esp_id].copy()
    esp_groups = sorted(esp_df['group_path'].unique())

    group_labels = [parse_path_meta(group_path)['label'] for group_path in esp_groups]

    train_groups, test_groups = train_test_split(
        esp_groups,
        test_size=0.2,
        random_state=random_state,
        stratify=group_labels,
        shuffle=True,
    )

    train_group_set = set(train_groups)
    test_group_set = set(test_groups)

    shared_groups = train_group_set & test_group_set
    if shared_groups:
        raise RuntimeError(f'Group leakage detected for {esp_id}: {sorted(shared_groups)}')

    train_files = sorted([fp for fp in esp_df['file_path'] if get_test_group_dir(fp) in train_group_set])
    test_files = sorted([fp for fp in esp_df['file_path'] if get_test_group_dir(fp) in test_group_set])

    esp_splits[esp_id] = {
        'train_groups': train_groups,
        'test_groups': test_groups,
        'train_files': train_files,
        'test_files': test_files,
    }

print(f'Total .data files: {len(all_files)}')
print(f'ESP ids found: {esp_ids}')
for esp_id in esp_ids:
    split = esp_splits[esp_id]
    print(
        f"{esp_id}: train_groups={len(split['train_groups'])}, "
        f"test_groups={len(split['test_groups'])}, "
        f"train_files={len(split['train_files'])}, test_files={len(split['test_files'])}"
    )

Total .data files: 2100
ESP ids found: ['dev1', 'dev2', 'dev3']
dev1: train_groups=560, test_groups=140, train_files=560, test_files=140
dev2: train_groups=560, test_groups=140, train_files=560, test_files=140
dev3: train_groups=560, test_groups=140, train_files=560, test_files=140


## Feature extraction per ESP

Global amplitude min/max считаются только на train и отдельно для каждой ESP.

In [4]:
def build_features_df(unit_cache: list[dict], global_min: float, global_max: float) -> pd.DataFrame:
    den = global_max - global_min
    if den <= 0:
        raise ValueError('Global max equals global min; cannot perform min-max scaling')

    rows = []
    for unit in unit_cache:
        file_path = unit['file_path']
        df = unit['df']
        signal = unit['filtered_signal']

        scaled_signal = (signal - global_min) / den
        scaled_signal = np.clip(scaled_signal, 0.0, 1.0)

        mean_val = float(np.mean(scaled_signal))
        std_val = float(np.std(scaled_signal))
        median_val = float(np.median(scaled_signal))
        min_val = float(np.min(scaled_signal))
        max_val = float(np.max(scaled_signal))
        q25 = float(np.percentile(scaled_signal, 25))
        q75 = float(np.percentile(scaled_signal, 75))
        iqr = q75 - q25
        range_val = max_val - min_val
        rms = float(np.sqrt(np.mean(scaled_signal ** 2)))
        energy = float(np.sum(scaled_signal ** 2))
        zcr = float(np.mean(np.abs(np.diff(np.signbit(scaled_signal - mean_val)).astype(np.int8))))

        sk = skewness(scaled_signal)
        kt = kurtosis_excess(scaled_signal)
        sc, ss, se, dom_freq, rolloff = spectral_features(scaled_signal)
        meta = parse_path_meta(file_path)

        rows.append({
            'file_path': str(file_path),
            'group_path': str(get_test_group_dir(file_path)),
            **meta,
            'n_packets': int(len(df)),
            'mean': mean_val,
            'std': std_val,
            'median': median_val,
            'skew': sk,
            'kurtosis': kt,
            'spectral_centroid': sc,
            'spectral_spread': ss,
            'spectral_entropy': se,
            'dominant_freq': dom_freq,
            'spectral_rolloff_85': rolloff,
            'min': min_val,
            'max': max_val,
            'q25': q25,
            'q75': q75,
            'iqr': iqr,
            'range': range_val,
            'rms': rms,
            'energy': energy,
            'zcr': zcr,
        })

    return pd.DataFrame(rows)


feature_cols = [
    'mean', 'std', 'median', 'skew', 'kurtosis',
    'spectral_centroid', 'spectral_spread', 'spectral_entropy',
    'dominant_freq', 'spectral_rolloff_85',
    'min', 'max', 'q25', 'q75', 'iqr', 'range',
    'rms', 'energy', 'zcr',
]

esp_artifacts: dict[str, dict] = {}
for esp_id in esp_ids:
    split = esp_splits[esp_id]
    train_unit_cache = build_unit_cache(split['train_files'])
    test_unit_cache = build_unit_cache(split['test_files'])

    global_min = min(float(np.min(unit['filtered_signal'])) for unit in train_unit_cache)
    global_max = max(float(np.max(unit['filtered_signal'])) for unit in train_unit_cache)

    train_features_df = build_features_df(train_unit_cache, global_min, global_max)
    test_features_df = build_features_df(test_unit_cache, global_min, global_max)

    esp_artifacts[esp_id] = {
        'global_min': global_min,
        'global_max': global_max,
        'train_features_df': train_features_df,
        'test_features_df': test_features_df,
    }

    print(
        f"{esp_id}: train_features={train_features_df.shape}, "
        f"test_features={test_features_df.shape}, "
        f"global_min={global_min:.6f}, global_max={global_max:.6f}"
    )

dev1: train_features=(560, 26), test_features=(140, 26), global_min=2.203125, global_max=29.085928
dev2: train_features=(560, 26), test_features=(140, 26), global_min=5.469660, global_max=37.418262
dev3: train_features=(560, 26), test_features=(140, 26), global_min=0.691347, global_max=10.399266


## PCA preprocessing per ESP

`StandardScaler` и `PCA` обучаются только на train и отдельно для каждой ESP.

In [5]:
for esp_id in esp_ids:
    train_features_df = esp_artifacts[esp_id]['train_features_df']
    test_features_df = esp_artifacts[esp_id]['test_features_df']

    X_train_raw = train_features_df[feature_cols].to_numpy(dtype=np.float64)
    X_test_raw = test_features_df[feature_cols].to_numpy(dtype=np.float64)

    feature_min = X_train_raw.min(axis=0)
    feature_max = X_train_raw.max(axis=0)

    X_train = np.clip(X_train_raw, feature_min, feature_max)
    X_test = np.clip(X_test_raw, feature_min, feature_max)

    scaler_pca = StandardScaler()
    X_train_std = scaler_pca.fit_transform(X_train)
    X_test_std = scaler_pca.transform(X_test)

    pca = PCA()
    X_train_pca = pca.fit_transform(X_train_std)
    X_test_pca = pca.transform(X_test_std)

    cum_evr = np.cumsum(pca.explained_variance_ratio_)
    k_95 = int(np.searchsorted(cum_evr, 0.95) + 1)

    X_train_pca_95 = X_train_pca[:, :k_95]
    X_test_pca_95 = X_test_pca[:, :k_95]

    y_train = (train_features_df['label'] != 'label_00').astype(int).to_numpy()
    y_test_file = (test_features_df['label'] != 'label_00').astype(int).to_numpy()

    esp_artifacts[esp_id].update({
        'feature_min': feature_min,
        'feature_max': feature_max,
        'scaler_pca': scaler_pca,
        'pca': pca,
        'k_95': int(k_95),
        'X_train_pca_95': X_train_pca_95,
        'X_test_pca_95': X_test_pca_95,
        'y_train': y_train,
        'y_test_file': y_test_file,
    })

    print(
        f"{esp_id}: k_95={k_95}, X_train_pca_95={X_train_pca_95.shape}, "
        f"X_test_pca_95={X_test_pca_95.shape}, train_class_1={(y_train == 1).sum()}, "
        f"test_class_1={(y_test_file == 1).sum()}"
    )

dev1: k_95=5, X_train_pca_95=(560, 5), X_test_pca_95=(140, 5), train_class_1=240, test_class_1=60
dev2: k_95=5, X_train_pca_95=(560, 5), X_test_pca_95=(140, 5), train_class_1=240, test_class_1=60
dev3: k_95=6, X_train_pca_95=(560, 6), X_test_pca_95=(140, 6), train_class_1=240, test_class_1=60


## Train one best model per ESP

Для каждой ESP обучается отдельная модель: `SVM(C=5.0, gamma='auto', kernel='rbf')`.

In [6]:
import sys
import subprocess
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Ensure CatBoost is available
try:
    from catboost import CatBoostClassifier
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost', '-q'])
    from catboost import CatBoostClassifier

# Define hyperparameter search spaces
model_spaces = {
    'CatBoost': (
        CatBoostClassifier(
            random_seed=42,
            loss_function='Logloss',
            verbose=False
        ),
        # {
        #     # Глубина деревьев (обычно 4–10, но для больших данных можно выше)
        #     'depth': [4, 6, 8, 10],
            
        #     # Скорость обучения (включая очень маленькие шаги)
        #     'learning_rate': [0.01, 0.05, 0.3],
            
        #     # Количество деревьев (итераций)
        #     'iterations': [100, 300, 800],
            
        #     # L2-регуляризация
        #     'l2_leaf_reg': [1, 5, 7],
            
        #     # Количество бинов для дискретизации признаков (чем больше, тем точнее, но дольше)
        #     'border_count': [128, 255, 512],
            
        #     # Доля случайно выбираемых признаков на каждом шаге (аналог colsample_bytree в XGBoost)
        #     'rsm': [0.5, 0.7, 0.9, 1.0],
            
        #     # Доля строк для обучения каждого дерева (субдискретизация)
        #     'subsample': [0.6, 0.8, 1.0],
            
        #     # Минимальное количество объектов в листе
        #     'min_data_in_leaf': [1, 3, 5, 10],
            
        #     # Метрика качества (можно добавить для контроля переобучения)
        #     'eval_metric': ['Logloss', 'AUC', 'Accuracy', 'F1'],
            
        #     # Ранняя остановка (не гиперпараметр, но полезно указать при поиске)
        #     # 'early_stopping_rounds': 50  # передаётся отдельно в .fit()
        # }
        {
            'depth': [4, 6, 8],
            'learning_rate': [0.03, 0.1],
            'iterations': [200, 400],
            'l2_leaf_reg': [3, 5]
        }
    ),
    # 'LogReg': (
    #     LogisticRegression(max_iter=5000, random_state=42),
    #     {
    #         'C': [0.1, 1.0, 10.0],
    #         'solver': ['lbfgs'],
    #         'penalty': ['l2']
    #     }
    # ),
    # 'RandomForest': (
    #     RandomForestClassifier(random_state=42, n_jobs=-1),
    #     {
    #         'n_estimators': [300, 600],
    #         'max_depth': [None, 12, 24],
    #         'min_samples_split': [2, 5],
    #         'min_samples_leaf': [1, 2]
    #     }
    # ),
    # 'SVM': (
    #     SVC(probability=True, random_state=42),
    #     {
    #         'C': [0.5, 1.0, 5.0, 10.0],  # расширенный диапазон
    #         'kernel': ['linear', 'rbf'],
    #         'gamma': ['scale', 'auto', 0.1, 0.01]  # добавлены числовые значения
    #     }
    # ),
}

# Main loop with hyperparameter tuning for each ESP
for esp_id in esp_ids:
    print(f'\n{"="*60}')
    print(f'Processing ESP: {esp_id}')
    print(f'{"="*60}')
    
    # Get data for this ESP
    X_train = esp_artifacts[esp_id]['X_train_pca_95']
    y_train = esp_artifacts[esp_id]['y_train']
    X_test = esp_artifacts[esp_id]['X_test_pca_95']
    y_test_file = esp_artifacts[esp_id]['y_test_file']
    
    # Get group labels for stratified group k-fold
    # Assuming you have train_files list and file_to_group mapping
    # If not available, use regular StratifiedKFold
    try:
        train_files = esp_artifacts[esp_id].get('train_files', None)
        file_to_group = esp_artifacts[esp_id].get('file_to_group', None)
        
        if train_files is not None and file_to_group is not None:
            train_cv_groups = [str(file_to_group[fp]) for fp in train_files]
            cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        else:
            # Fallback to regular stratified k-fold
            from sklearn.model_selection import StratifiedKFold
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            train_cv_groups = None
    except:
        from sklearn.model_selection import StratifiedKFold
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        train_cv_groups = None
    
    # Store results for this ESP
    search_results = []
    best_search = None
    best_model = None
    
    # Grid search for each model type
    for model_name, (estimator, param_grid) in model_spaces.items():
        print(f'\n  Grid search for: {model_name}')
        
        try:
            search = GridSearchCV(
                estimator=estimator,
                param_grid=param_grid,
                scoring='f1',
                cv=cv,
                n_jobs=-1,
                refit=True
            )
            
            # Fit with or without groups
            if train_cv_groups is not None:
                search.fit(X_train, y_train, groups=train_cv_groups)
            else:
                search.fit(X_train, y_train)
            
            # Evaluate on test set
            best_model_candidate = search.best_estimator_
            y_pred = best_model_candidate.predict(X_test)
            
            if hasattr(best_model_candidate, 'predict_proba'):
                y_proba = best_model_candidate.predict_proba(X_test)[:, 1]
            else:
                # Fallback for models without predict_proba
                y_score = best_model_candidate.decision_function(X_test)
                y_proba = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-12)
            
            result = {
                'model': model_name,
                'best_cv_f1': search.best_score_,
                'test_f1': f1_score(y_test_file, y_pred),
                'test_accuracy': accuracy_score(y_test_file, y_pred),
                'test_roc_auc': roc_auc_score(y_test_file, y_proba),
                'best_params': search.best_params_
            }
            search_results.append(result)
            
            # Track best model overall for this ESP
            if best_search is None or search.best_score_ > best_search.best_score_:
                best_search = search
                best_model = best_model_candidate
            
            print(f'    Best CV F1: {search.best_score_:.4f}')
            print(f'    Test F1: {result["test_f1"]:.4f}')
            
        except Exception as e:
            print(f'    Error with {model_name}: {str(e)}')
            continue
    
    # Create results dataframe and rank models
    if search_results:
        results_df = pd.DataFrame(search_results).sort_values('best_cv_f1', ascending=False).reset_index(drop=True)
        
        print(f'\n  Model ranking for {esp_id} (by best CV F1):')
        print(results_df[['model', 'best_cv_f1', 'test_f1', 'test_accuracy', 'test_roc_auc']].to_string(index=False))
        
        # Get best model info
        best_model_name = results_df.loc[0, 'model']
        print(f'\n  Best model: {best_model_name}')
        print(f'  Best CV F1: {best_search.best_score_:.4f}')
        print(f'  Best params: {best_search.best_params_}')
        
        # Final predictions with best model
        best_pred = best_model.predict(X_test)
        
        print(f'\n  File-level test metrics (best model for {esp_id}):')
        print(f'  accuracy: {accuracy_score(y_test_file, best_pred):.4f}')
        print(classification_report(
            y_test_file,
            best_pred,
            labels=[0, 1],
            target_names=['static / no motion', 'dynamic / motion'],
            zero_division=0,
        ))
        
        # Store results in esp_artifacts
        esp_artifacts[esp_id].update({
            'model': best_model,
            'file_pred': best_pred,
            'hyperparameter_search_results': results_df,
            'best_model_name': best_model_name,
            'best_model_params': best_search.best_params_
        })
        
        # Also store predictions from all models if needed
        esp_artifacts[esp_id]['all_models_predictions'] = {
            row['model']: {
                'best_cv_f1': row['best_cv_f1'],
                'test_f1': row['test_f1'],
                'test_accuracy': row['test_accuracy'],
                'test_roc_auc': row['test_roc_auc']
            }
            for _, row in results_df.iterrows()
        }
        
    else:
        print(f'\n  WARNING: No successful models for {esp_id}')
        # Fallback to original SVC with default params
        print(f'  Using default SVC as fallback')
        model = SVC(C=5.0, gamma='auto', kernel='rbf', probability=True, random_state=42)
        model.fit(X_train, y_train)
        file_pred = model.predict(X_test)
        
        esp_artifacts[esp_id].update({
            'model': model,
            'file_pred': file_pred,
            'hyperparameter_search_results': None,
            'best_model_name': 'SVC_fallback'
        })
        
        print(f'  accuracy: {accuracy_score(y_test_file, file_pred):.4f}')
        print(classification_report(
            y_test_file,
            file_pred,
            labels=[0, 1],
            target_names=['static / no motion', 'dynamic / motion'],
            zero_division=0,
        ))


Processing ESP: dev1

  Grid search for: CatBoost
    Best CV F1: 0.9938
    Test F1: 1.0000

  Model ranking for dev1 (by best CV F1):
   model  best_cv_f1  test_f1  test_accuracy  test_roc_auc
CatBoost    0.993771      1.0            1.0           1.0

  Best model: CatBoost
  Best CV F1: 0.9938
  Best params: {'depth': 4, 'iterations': 200, 'l2_leaf_reg': 5, 'learning_rate': 0.03}

  File-level test metrics (best model for dev1):
  accuracy: 1.0000
                    precision    recall  f1-score   support

static / no motion       1.00      1.00      1.00        80
  dynamic / motion       1.00      1.00      1.00        60

          accuracy                           1.00       140
         macro avg       1.00      1.00      1.00       140
      weighted avg       1.00      1.00      1.00       140


Processing ESP: dev2

  Grid search for: CatBoost
    Best CV F1: 0.9689
    Test F1: 0.9756

  Model ranking for dev2 (by best CV F1):
   model  best_cv_f1  test_f1  test_accurac

## Save notebook artifacts per ESP

`run_inference` из `tools/classic_ml_inference.py` используется для каждой ESP-модели отдельно, поэтому сохраняются отдельные `model/preprocessing/metadata` в подпапки `esp_*`.

In [7]:
artifacts_root = project_root / 'artifacts' / 'one_person_classic_ml_majority_vote_metrics_each_esp'
artifacts_root.mkdir(parents=True, exist_ok=True)

for esp_id in esp_ids:
    esp_dir = artifacts_root / esp_id
    esp_dir.mkdir(parents=True, exist_ok=True)

    model_path = esp_dir / 'model.joblib'
    preprocessing_path = esp_dir / 'preprocessing.joblib'
    metadata_path = esp_dir / 'metadata.json'

    best_model = esp_artifacts[esp_id]['model']
    joblib.dump(best_model, model_path)

    preproc_bundle = {
        'median_width': median_width,
        'global_min': float(esp_artifacts[esp_id]['global_min']),
        'global_max': float(esp_artifacts[esp_id]['global_max']),
        'feature_cols': feature_cols,
        'feature_min': esp_artifacts[esp_id]['feature_min'],
        'feature_max': esp_artifacts[esp_id]['feature_max'],
        'scaler_pca': esp_artifacts[esp_id]['scaler_pca'],
        'pca': esp_artifacts[esp_id]['pca'],
        'k_95': int(esp_artifacts[esp_id]['k_95']),
        'model_type': 'sklearn',
        'esp_id': esp_id,
    }
    joblib.dump(preproc_bundle, preprocessing_path)

    results_df = esp_artifacts[esp_id].get('hyperparameter_search_results')
    best_cv_score_f1 = None
    if results_df is not None and len(results_df) > 0 and 'best_cv_f1' in results_df.columns:
        best_cv_score_f1 = float(results_df.iloc[0]['best_cv_f1'])

    metadata = {
        'source_notebook': 'each_esp_has_model_majority_vote_metrics.ipynb',
        'model': str(esp_artifacts[esp_id].get('best_model_name', type(best_model).__name__)),
        'selection': 'GridSearchCV per ESP',
        'esp_id': esp_id,
        'best_params': esp_artifacts[esp_id].get('best_model_params', {}),
        'best_cv_score_f1': best_cv_score_f1,
        'split': 'group train/test split by test_N folders, per ESP, test_size=0.2, random_state=42',
        'model_path': str(model_path),
        'preprocessing_path': str(preprocessing_path),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

    esp_artifacts[esp_id].update({
        'model_path': model_path,
        'preprocessing_path': preprocessing_path,
        'metadata_path': metadata_path,
    })

    print(f'[{esp_id}] model_path: {model_path}')
    print(f'[{esp_id}] preprocessing_path: {preprocessing_path}')
    print(f'[{esp_id}] metadata_path: {metadata_path}')

[dev1] model_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority_vote_metrics_each_esp/dev1/model.joblib
[dev1] preprocessing_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority_vote_metrics_each_esp/dev1/preprocessing.joblib
[dev1] metadata_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority_vote_metrics_each_esp/dev1/metadata.json
[dev2] model_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority_vote_metrics_each_esp/dev2/model.joblib
[dev2] preprocessing_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority_vote_metrics_each_esp/dev2/preprocessing.joblib
[dev2] metadata_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority_vote_metrics_each_esp/dev2/metadata.json
[dev3] model_path: /home/gleb/learning/CSI-activity-detection/artifacts/one_person_classic_ml_majority

## Majority vote across 3 ESP models

Для каждого `test_N` каждая ESP-модель даёт свой итоговый класс (внутри своей ESP сначала majority vote по её `.data` файлам).

Затем финальная метка `test_N` выбирается majority vote по трём ответам (`dev1`, `dev2`, `dev3`).

In [8]:
def true_class_from_label(label_name: str) -> int:
    return 0 if label_name == 'label_00' else 1


esp_group_vote_rows = []

for esp_id in esp_ids:
    split = esp_splits[esp_id]
    model_path = esp_artifacts[esp_id]['model_path']
    preprocessing_path = esp_artifacts[esp_id]['preprocessing_path']

    test_groups_sorted = sorted(split['test_groups'])

    for idx, test_group in enumerate(test_groups_sorted, start=1):
        data_files = sorted([fp for fp in split['test_files'] if get_test_group_dir(fp) == test_group])
        if not data_files:
            continue

        file_results = [run_inference(model_path, preprocessing_path, data_file) for data_file in data_files]
        file_predictions = [int(result['predicted_class']) for result in file_results]

        votes_static = file_predictions.count(0)
        votes_dynamic = file_predictions.count(1)
        esp_pred = 1 if votes_dynamic >= votes_static else 0

        group_label = parse_path_meta(test_group)['label']
        y_true = true_class_from_label(group_label)

        esp_group_vote_rows.append({
            'group_path': str(test_group),
            'test_id': test_group.name,
            'label': group_label,
            'y_true': y_true,
            'esp_id': esp_id,
            'n_files': len(data_files),
            'esp_votes_static': votes_static,
            'esp_votes_dynamic': votes_dynamic,
            'esp_pred': esp_pred,
        })

        if idx % 50 == 0:
            print(f'[{esp_id}] processed test groups: {idx}/{len(test_groups_sorted)}')

esp_group_preds_df = pd.DataFrame(esp_group_vote_rows)

group_rows = []
for group_path, group_df in esp_group_preds_df.groupby('group_path', sort=True):
    preds = [int(x) for x in group_df['esp_pred'].tolist()]
    n_esp_votes = len(preds)

    if n_esp_votes < len(esp_ids):
        continue

    votes_static = preds.count(0)
    votes_dynamic = preds.count(1)
    y_pred_final = 1 if votes_dynamic >= votes_static else 0

    first = group_df.iloc[0]
    group_rows.append({
        'group_path': group_path,
        'test_id': first['test_id'],
        'label': first['label'],
        'y_true': int(first['y_true']),
        'n_esp_votes': n_esp_votes,
        'final_votes_static': votes_static,
        'final_votes_dynamic': votes_dynamic,
        'y_pred_final': y_pred_final,
    })

final_vote_results_df = pd.DataFrame(group_rows).sort_values('group_path').reset_index(drop=True)

print(f'ESP-level group votes: {len(esp_group_preds_df)} rows')
print(f'Final groups scored with 3-model majority vote: {len(final_vote_results_df)}')
final_vote_results_df.head()

[dev1] processed test groups: 50/140
[dev1] processed test groups: 100/140
[dev2] processed test groups: 50/140
[dev2] processed test groups: 100/140
[dev3] processed test groups: 50/140
[dev3] processed test groups: 100/140
ESP-level group votes: 420 rows
Final groups scored with 3-model majority vote: 140


,group_path,test_id,label,y_true,n_esp_votes,final_votes_static,final_votes_dynamic,y_pred_final
0,/home/gleb/learning/CSI-activity-detection/wif...,test_02,label_00,0,3,3,0,0
1,/home/gleb/learning/CSI-activity-detection/wif...,test_13,label_00,0,3,3,0,0
2,/home/gleb/learning/CSI-activity-detection/wif...,test_20,label_00,0,3,3,0,0
3,/home/gleb/learning/CSI-activity-detection/wif...,test_21,label_00,0,3,3,0,0
4,/home/gleb/learning/CSI-activity-detection/wif...,test_34,label_00,0,3,3,0,0


## Test metrics after final cross-ESP majority vote

In [9]:
y_test_group = final_vote_results_df['y_true'].to_numpy()
y_pred_group = final_vote_results_df['y_pred_final'].to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_group,
    y_pred_group,
    labels=[0, 1],
    zero_division=0,
)

cm = confusion_matrix(y_test_group, y_pred_group, labels=[0, 1])
acc = accuracy_score(y_test_group, y_pred_group)

final_metrics_df = pd.DataFrame({
    'class': ['static / no motion', 'dynamic / motion'],
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'support': support,
})

final_summary_df = pd.DataFrame([{
    'accuracy': acc,
    'n_test_groups': int(len(final_vote_results_df)),
    'n_models_in_vote': int(len(esp_ids)),
}])

print('Confusion matrix (rows=true, cols=pred):')
print(cm)
print()
print(classification_report(
    y_test_group,
    y_pred_group,
    labels=[0, 1],
    target_names=['static / no motion', 'dynamic / motion'],
    zero_division=0,
))
print()
print('Final majority-vote summary:')
final_summary_df

Confusion matrix (rows=true, cols=pred):
[[80  0]
 [ 0 60]]

                    precision    recall  f1-score   support

static / no motion       1.00      1.00      1.00        80
  dynamic / motion       1.00      1.00      1.00        60

          accuracy                           1.00       140
         macro avg       1.00      1.00      1.00       140
      weighted avg       1.00      1.00      1.00       140


Final majority-vote summary:


,accuracy,n_test_groups,n_models_in_vote
0,1.0,140,3


## Test errors after final cross-ESP majority vote

In [10]:
final_errors_df = final_vote_results_df[final_vote_results_df['y_true'] != final_vote_results_df['y_pred_final']].copy()

print(f'Final majority-vote test group errors: {len(final_errors_df)}')
final_errors_df

Final majority-vote test group errors: 0


,group_path,test_id,label,y_true,n_esp_votes,final_votes_static,final_votes_dynamic,y_pred_final
